# 1. Model Training With Class x Skin-Tone Class-Balanced Cross-Entropy

This notebook implements the custom-loss experiment retained in the thesis. The weighting scheme is defined over `class x skin_tone` pairs only.

The objective is to evaluate the selected dermoscopic ViT baseline for the 11-class and binary tasks using the beta values retained in the thesis: `0.999` for `11f` and `0.99` for `2f`. Sex and age are still used for evaluation, but they are not part of the custom-loss weighting scheme.

## 2. Global Imports

All core imports are centralized here.


In [ ]:
import copy
import gc
import os
import random
import warnings
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm


## 3. Global Configuration

This cell restricts the experiment to the dermoscopic ViT configuration used in the final supervised intervention experiments. The custom loss uses class x skin-tone effective-number weights.

In [ ]:
MODELS_TO_RUN = ["ViT"]
TASKS_TO_RUN = ["11f", "2f"]
SENSORS_TO_RUN = ["dermoscopic"]

LRS_BY_MODEL = {
    "ViT": [3e-5],
}

BATCH_SIZES_BY_TASK = {
    "11f": [32],
    "2f": [16],
}

BETA_VALUES_BY_TASK = {
    "11f": [0.999],
    "2f": [0.99],
}

EARLY_STOPPING_PATIENCE_BY_MODEL = {
    "ViT": 3,
}

N_RUNS_BY_MODEL = {
    "ViT": 3,
}

EPOCHS = 100
ALPHA = 0.40
BASE_SEED = 42
RESUME = True

NUM_WORKERS = 0  # Required in this notebook: IndexedImageFolder + Windows/Jupyter multiprocessing raises WinError 5 in this environment
PIN_MEMORY = torch.cuda.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

EXPERIMENT_ROOT = Path("Experiment_2") / "custom_loss_class_skin_tone"
MODELS_ROOT = EXPERIMENT_ROOT / "models"
REPORTS_ROOT = EXPERIMENT_ROOT / "reports"
RUNS_ROOT = EXPERIMENT_ROOT / "runs"

for path in [MODELS_ROOT, REPORTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_ROOTS = {
    ("11f", "dermoscopic"): Path("data") / "dataset_11f" / "dermoscopic",
    ("2f", "dermoscopic"): Path("data") / "dataset_2f" / "dermoscopic",
}

EXPECTED_CLASSES = {
    "11f": 11,
    "2f": 2,
}

BASELINE_EXPERIMENT_1_AVERAGES = (
    Path("Experiment_1") / "training_runs_final_fixed" / "runs" / "ViT" / "11f" / "dermoscopic" / "averages.csv"
)
BASELINE_EXPERIMENT_1_LR = 3e-5
BASELINE_EXPERIMENT_1_BATCH_SIZE = 32

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN and hf_login is not None:
    try:
        hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass
else:
    warnings.filterwarnings(
        "ignore",
        message=".*unauthenticated requests to the HF Hub.*",
    )

print("Device:", DEVICE)
print("Experiment root:", EXPERIMENT_ROOT)
print("HF token detected:", bool(HF_TOKEN))
print("Alpha for FP Score:", ALPHA)
print("Betas by task:", BETA_VALUES_BY_TASK)
print("\nLearning rates by model:")
for model_name, lr_values in LRS_BY_MODEL.items():
    print(f"- {model_name}: {lr_values}")

print("\nBatch sizes by task:")
for task_name, batch_sizes in BATCH_SIZES_BY_TASK.items():
    print(f"- {task_name}: {batch_sizes}")


## 4. Metadata Preparation

This section loads the training metadata and prepares the subgroup variables that will later be used for evaluation and fairness analysis.

The metadata are indexed by `isic_id` and reduced to the variables used later in the notebook: `sex`, `skin_tone`, and `age`.

The `sex` variable is kept simple because the metadata only contain `male` and `female`, with no missing values. For `age`, missing values are assigned to `unknown`, since `age_approx` does contain missing values in the metadata.


In [ ]:
metadata_path = Path("data/raw/MILK10k_Training_Metadata.csv")
raw_metadata_df = pd.read_csv(
    metadata_path,
    usecols=["isic_id", "skin_tone_class", "sex", "age_approx"],
).copy()

metadata_df = pd.DataFrame({
    "isic_id": raw_metadata_df["isic_id"],
    "sex": raw_metadata_df["sex"].fillna("unknown").astype(str).str.strip().str.lower(),
    "skin_tone": raw_metadata_df["skin_tone_class"].fillna(-1).astype(int),
})

metadata_df["age"] = raw_metadata_df["age_approx"].apply(
    lambda value: "unknown" if pd.isna(value) else int(value)
)

SEX_TO_ID = {"male": 0, "female": 1, "unknown": 2}

isic_to_skin = dict(zip(metadata_df["isic_id"], metadata_df["skin_tone"]))
isic_to_sex = dict(zip(metadata_df["isic_id"], metadata_df["sex"]))
isic_to_age = dict(zip(metadata_df["isic_id"], metadata_df["age"]))


def normalize_sex(value: str) -> str:
    value = (value or "unknown").strip().lower()
    return value if value in SEX_TO_ID else "unknown"


def normalize_age(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "unknown"
    if isinstance(value, str):
        stripped = value.strip().lower()
        if stripped == "unknown" or stripped == "":
            return "unknown"
        try:
            return int(float(stripped))
        except Exception:
            return "unknown"
    try:
        return int(value)
    except Exception:
        return "unknown"


def format_lr(lr: float) -> str:
    return f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def format_float_tag(value: float) -> str:
    return f"{float(value):g}".replace(".", "p")


def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(name).strip().lower()).strip("_")


print("metadata_df:", metadata_df.shape)
metadata_df.head()


## 5. Transforms and Dataset Loading

This section defines the preprocessing pipeline and the helper functions used to load `train`, `validation`, and `test` datasets from the exported folders.


In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


IMAGE_SIZE = 224

class IndexedImageFolder(ImageFolder):
    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        return image, label, index

DATA_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
])


def load_datasets(task: str, sensor: str):
    dataset_root = DATASET_ROOTS[(task, sensor)]

    train_dataset = IndexedImageFolder(dataset_root / "train", transform=DATA_TRANSFORM)
    val_dataset = ImageFolder(dataset_root / "validation", transform=DATA_TRANSFORM)
    test_dataset = ImageFolder(dataset_root / "test", transform=DATA_TRANSFORM)

    class_names = train_dataset.classes
    if len(class_names) != EXPECTED_CLASSES[task]:
        raise RuntimeError(
            f"Expected {EXPECTED_CLASSES[task]} classes for {task}, but found {len(class_names)} in {dataset_root}."
        )

    if val_dataset.classes != class_names or test_dataset.classes != class_names:
        raise RuntimeError("Class folders are not consistent across train, validation, and test.")

    return train_dataset, val_dataset, test_dataset, class_names


def make_dataloaders(train_dataset, val_dataset, test_dataset, batch_size: int):
    persistent = NUM_WORKERS > 0

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )

    return train_loader, val_loader, test_loader


set_all_seeds(BASE_SEED)
sample_task = TASKS_TO_RUN[0]
sample_sensor = SENSORS_TO_RUN[0]
sample_train_dataset, sample_val_dataset, sample_test_dataset, sample_class_names = load_datasets(sample_task, sample_sensor)

print(f"Sample dataset: task={sample_task}, sensor={sample_sensor}")
print(f"- train: {len(sample_train_dataset)} images")
print(f"- validation: {len(sample_val_dataset)} images")
print(f"- test: {len(sample_test_dataset)} images")
print(f"- classes ({len(sample_class_names)}): {sample_class_names}")


## 6. Model Factory

This section defines the model-construction helpers used in the experiments.


In [ ]:
def cleanup_memory(*objs):
    for obj in objs:
        try:
            if hasattr(obj, "to"):
                obj.to("cpu")
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def create_model(model_name: str, num_classes: int):
    if model_name == "Resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        return model.to(DEVICE)

    if model_name == "ViT":
        try:
            import timm
        except Exception as error:
            raise ImportError("timm is required for ViT. Install it before running this notebook.") from error

        model = timm.create_model(
            "vit_base_patch32_224",
            pretrained=True,
            num_classes=num_classes,
        )
        return model.to(DEVICE)

    raise ValueError(f"Unsupported model: {model_name}")


sample_model_name = MODELS_TO_RUN[0]
sample_num_classes = EXPECTED_CLASSES[TASKS_TO_RUN[0]]
sample_model = create_model(sample_model_name, sample_num_classes)

print(f"Sample model: {sample_model_name}")
print(f"Number of classes: {sample_num_classes}")
print(f"Device: {DEVICE}")


## 7. Weight Calculation, Training, and Evaluation

This section computes the class-balanced weights over `class x skin_tone` pairs and then defines the shared training and evaluation helpers used by the experiment.

Sex and age are not used for the loss weights. They are kept only as sensitive attributes for the Equalized Odds evaluation.

In [ ]:
def compute_f1(y_true, y_pred, num_classes: int) -> float:
    if num_classes == 2:
        return float(f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def compute_auc(y_true, y_prob, num_classes: int) -> float:
    try:
        if num_classes == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def compute_tpr_fpr_by_class(y_true, y_pred, class_names):
    rows = []
    tpr_values = []
    fpr_values = []

    for idx, class_name in enumerate(class_names):
        y_true_bin = y_true == idx
        y_pred_bin = y_pred == idx

        tp = np.sum(y_true_bin & y_pred_bin)
        fn = np.sum(y_true_bin & (~y_pred_bin))
        fp = np.sum((~y_true_bin) & y_pred_bin)
        tn = np.sum((~y_true_bin) & (~y_pred_bin))

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        rows.append({
            "class": class_name,
            "tpr": float(tpr) if not np.isnan(tpr) else np.nan,
            "fpr": float(fpr) if not np.isnan(fpr) else np.nan,
        })
        tpr_values.append(tpr)
        fpr_values.append(fpr)

    summary = {
        "avg_tpr": float(np.nanmean(tpr_values)) if len(tpr_values) else np.nan,
        "avg_fpr": float(np.nanmean(fpr_values)) if len(fpr_values) else np.nan,
    }
    return pd.DataFrame(rows), summary


def compute_eo_gap(y_true, y_pred, sensitive, num_classes: int, ignore_values=None):
    """Compute a multiclass Equalized Odds Gap using both TPR and FPR."""
    sensitive = np.asarray(sensitive)
    if ignore_values is not None:
        ignore_values = set(ignore_values)
        keep = np.asarray([value not in ignore_values for value in sensitive], dtype=bool)
        y_true = y_true[keep]
        y_pred = y_pred[keep]
        sensitive = sensitive[keep]

    groups = sorted(set(sensitive.tolist()))
    if len(groups) <= 1:
        return 0.0

    class_gaps = []
    for class_idx in range(num_classes):
        tprs = []
        fprs = []
        for group in groups:
            idx = sensitive == group
            yt = y_true[idx] == class_idx
            yp = y_pred[idx] == class_idx
            tp = np.sum(yt & yp)
            fn = np.sum(yt & (~yp))
            fp = np.sum((~yt) & yp)
            tn = np.sum((~yt) & (~yp))
            tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            tprs.append(tpr)
            fprs.append(fpr)

        mean_tpr = np.nanmean(tprs)
        mean_fpr = np.nanmean(fprs)
        mean_tpr = 0.0 if np.isnan(mean_tpr) else mean_tpr
        mean_fpr = 0.0 if np.isnan(mean_fpr) else mean_fpr

        tpr_gaps = [abs((mean_tpr if np.isnan(tpr) else tpr) - mean_tpr) for tpr in tprs]
        fpr_gaps = [abs((mean_fpr if np.isnan(fpr) else fpr) - mean_fpr) for fpr in fprs]
        class_gap = 0.5 * (float(np.mean(tpr_gaps)) + float(np.mean(fpr_gaps)))
        class_gaps.append(class_gap)

    return float(np.mean(class_gaps)) if class_gaps else 0.0


def compute_G(eo_gap_age: float, eo_gap_sex: float, eo_gap_skin: float) -> float:
    return float(np.mean([1 - eo_gap_age, 1 - eo_gap_sex, 1 - eo_gap_skin]))


def compute_metric_A(auc: float, g_value: float, alpha: float = ALPHA) -> float:
    return float(alpha * auc + (1 - alpha) * g_value)


def compute_class_counts(dataset):
    return np.bincount(dataset.targets, minlength=len(dataset.classes)).astype(np.int64)


def compute_cb_weights(class_counts, beta: float):
    class_counts = np.asarray(class_counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, class_counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
    weights = weights / weights.sum() * len(class_counts)
    return weights.astype(np.float32)


def compute_sample_sensitive_attributes(dataset):
    skin_values = []
    sex_values = []
    age_values = []
    for path, _ in dataset.samples:
        isic_id = Path(path).stem
        skin_values.append(int(isic_to_skin.get(isic_id, -1)))
        sex_values.append(normalize_sex(isic_to_sex.get(isic_id, "unknown")))
        age_values.append(normalize_age(isic_to_age.get(isic_id, "unknown")))
    return (
        np.asarray(skin_values, dtype=np.int32),
        np.asarray(sex_values, dtype=object),
        np.asarray(age_values, dtype=object),
    )


def compute_class_skin_tone_weights(train_dataset, beta: float):
    targets = np.asarray(train_dataset.targets, dtype=np.int64)
    skin_values, _, _ = compute_sample_sensitive_attributes(train_dataset)
    class_names = list(train_dataset.classes)

    class_counts = compute_class_counts(train_dataset)
    sample_weights = np.zeros(len(train_dataset), dtype=np.float32)
    combo_rows = []

    combo_counts = {}
    for sample_idx, class_idx in enumerate(targets):
        skin_tone = int(skin_values[sample_idx])
        if skin_tone <= 0:
            continue
        combo_key = (int(class_idx), skin_tone)
        combo_counts[combo_key] = combo_counts.get(combo_key, 0) + 1

    if not combo_counts:
        raise RuntimeError("No valid class-skin_tone combinations were found in the training dataset.")

    combo_items = list(combo_counts.items())
    raw_combo_weights = np.zeros(len(combo_items), dtype=np.float64)
    for i, (_, combo_count) in enumerate(combo_items):
        effective_num = 1.0 - np.power(beta, combo_count)
        raw_combo_weights[i] = (1.0 - beta) / max(effective_num, 1e-12)

    positive_mask = raw_combo_weights > 0
    normalization = raw_combo_weights[positive_mask].mean() if np.any(positive_mask) else 1.0
    normalization = 1.0 if normalization == 0 else normalization
    normalized_combo_weights = raw_combo_weights / normalization

    combo_weight_lookup = {}
    for i, (combo_key, combo_count) in enumerate(combo_items):
        class_idx, skin_tone = combo_key
        final_weight = float(normalized_combo_weights[i])
        combo_weight_lookup[combo_key] = final_weight
        combo_rows.append({
            "class": class_names[class_idx],
            "skin_tone": int(skin_tone),
            "class_count": int(class_counts[class_idx]),
            "combo_count": int(combo_count),
            "combo_weight": final_weight,
        })

    for sample_idx, class_idx in enumerate(targets):
        skin_tone = int(skin_values[sample_idx])
        combo_key = (int(class_idx), skin_tone)
        sample_weights[sample_idx] = combo_weight_lookup.get(combo_key, 0.0)

    combo_df = pd.DataFrame(combo_rows).sort_values(
        by=["class", "combo_count", "skin_tone"],
        ascending=[True, False, True],
    ).reset_index(drop=True)
    return class_counts, combo_df, sample_weights.astype(np.float32), combo_df.copy()


class SampleWeightedCrossEntropy(nn.Module):
    def __init__(self, sample_weights):
        super().__init__()
        self.register_buffer("sample_weights", torch.tensor(sample_weights, dtype=torch.float32))

    def forward(self, logits, targets, sample_indices):
        losses = F.cross_entropy(logits, targets, reduction="none")
        weights = self.sample_weights[sample_indices]
        return (losses * weights).mean()


CLASS_SKIN_TONE_WEIGHT_CACHE = {}


def create_cb_sensitive_cross_entropy_loss(train_dataset, beta: float, cache_key=None):
    if cache_key is not None and cache_key in CLASS_SKIN_TONE_WEIGHT_CACHE:
        class_counts, combo_df, sample_weights, preview_df = CLASS_SKIN_TONE_WEIGHT_CACHE[cache_key]
    else:
        class_counts, combo_df, sample_weights, preview_df = compute_class_skin_tone_weights(train_dataset, beta=beta)
        if cache_key is not None:
            CLASS_SKIN_TONE_WEIGHT_CACHE[cache_key] = (class_counts, combo_df.copy(), sample_weights, preview_df.copy())
    loss_fn = SampleWeightedCrossEntropy(sample_weights).to(DEVICE)
    return loss_fn, class_counts, combo_df.copy(), sample_weights, preview_df.copy()


def get_predictions_targets_probs(model, loader):
    model.eval()
    y_true_all, y_pred_all, y_prob_all = [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=PIN_MEMORY)
            logits = model(images)
            y_true_all.append(labels.numpy())
            y_pred_all.append(logits.argmax(dim=1).cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(y_true_all), np.concatenate(y_pred_all), np.concatenate(y_prob_all)


def age_sort_key(value):
    if value == "unknown":
        return (1, float("inf"))
    return (0, int(value))


def build_sensitive_arrays(dataset):
    skin_vals, sex_vals, age_vals = [], [], []
    for path, _ in dataset.samples:
        isic_id = Path(path).stem
        skin_vals.append(int(isic_to_skin.get(isic_id, -1)))
        sex_vals.append(SEX_TO_ID[normalize_sex(isic_to_sex.get(isic_id, "unknown"))])
        age_vals.append(normalize_age(isic_to_age.get(isic_id, "unknown")))
    unique_age_groups = sorted(set(age_vals), key=age_sort_key)
    age_map = {value: idx for idx, value in enumerate(unique_age_groups)}
    age_ids = np.array([age_map[value] for value in age_vals], dtype=np.int32)
    return np.array(skin_vals, dtype=np.int32), np.array(sex_vals, dtype=np.int32), age_ids


def load_baseline_11f_per_class_accuracy():
    if not BASELINE_EXPERIMENT_1_AVERAGES.exists():
        return None
    baseline_df = pd.read_csv(BASELINE_EXPERIMENT_1_AVERAGES)
    if baseline_df.empty:
        return None
    baseline_df["lr"] = pd.to_numeric(baseline_df["lr"], errors="coerce")
    baseline_df["batch_size"] = pd.to_numeric(baseline_df["batch_size"], errors="coerce")
    row = baseline_df[(np.isclose(baseline_df["lr"], BASELINE_EXPERIMENT_1_LR)) & (baseline_df["batch_size"] == BASELINE_EXPERIMENT_1_BATCH_SIZE)]
    if row.empty:
        return None
    row = row.iloc[0]
    baseline = {"val": {}, "test": {}}
    for key, value in row.items():
        if key.startswith("val_acc_"):
            baseline["val"][key.replace("val_", "", 1)] = float(value)
        elif key.startswith("test_acc_"):
            baseline["test"][key.replace("test_", "", 1)] = float(value)
    return baseline


def compute_metric_B(split_metrics, baseline_split_metrics, class_names):
    if baseline_split_metrics is None:
        return np.nan, np.nan, np.nan
    improved_count = 0
    deltas = []
    for class_name in class_names:
        metric_key = f"acc_{safe_name(class_name)}"
        current_value = split_metrics.get(metric_key, np.nan)
        baseline_value = baseline_split_metrics.get(metric_key, np.nan)
        if np.isnan(current_value) or np.isnan(baseline_value):
            continue
        delta = float(current_value - baseline_value)
        deltas.append(delta)
        if delta > 0:
            improved_count += 1
    if not deltas:
        return np.nan, np.nan, np.nan
    metric_b = improved_count / len(deltas)
    mean_delta = float(np.mean(deltas))
    return float(metric_b), int(improved_count), mean_delta
def evaluate_split(model, loader, dataset, class_names, task, baseline_split_metrics=None):
    y_true, y_pred, y_prob = get_predictions_targets_probs(model, loader)
    num_classes = len(class_names)
    skin_arr, sex_arr, age_arr = build_sensitive_arrays(dataset)
    metrics = {"accuracy": float(accuracy_score(y_true, y_pred)), "f1": compute_f1(y_true, y_pred, num_classes), "auc": compute_auc(y_true, y_prob, num_classes)}
    metrics["eo_gap_skin"] = compute_eo_gap(y_true, y_pred, skin_arr, num_classes, ignore_values={0, -1})
    metrics["eo_gap_sex"] = compute_eo_gap(y_true, y_pred, sex_arr, num_classes)
    metrics["eo_gap_age"] = compute_eo_gap(y_true, y_pred, age_arr, num_classes)
    metrics["G"] = compute_G(metrics["eo_gap_age"], metrics["eo_gap_sex"], metrics["eo_gap_skin"])
    metrics["objective"] = compute_metric_A(metrics["auc"], metrics["G"], ALPHA)
    metrics["metric_A"] = metrics["objective"]
    tpr_fpr_df, tpr_fpr_summary = compute_tpr_fpr_by_class(y_true, y_pred, class_names)
    metrics["avg_tpr"] = tpr_fpr_summary["avg_tpr"]
    metrics["avg_fpr"] = tpr_fpr_summary["avg_fpr"]
    per_class_rows = []
    for idx, class_name in enumerate(class_names):
        mask = y_true == idx
        class_acc = float(accuracy_score(y_true[mask], y_pred[mask])) if np.sum(mask) else np.nan
        per_class_rows.append({"class": class_name, "n": int(np.sum(mask)), "accuracy": class_acc})
        metrics[f"acc_{safe_name(class_name)}"] = class_acc
    metric_b, improved_count, mean_delta = compute_metric_B(metrics, baseline_split_metrics if task == "11f" else None, class_names)
    metrics["metric_B"] = metric_b
    metrics["metric_B_n_improved"] = improved_count
    metrics["metric_B_mean_delta"] = mean_delta
    per_class_df = pd.DataFrame(per_class_rows).merge(tpr_fpr_df, on="class", how="left")

    for _, row in tpr_fpr_df.iterrows():
        class_key = safe_name(row["class"])
        metrics[f"tpr_{class_key}"] = row["tpr"]
        metrics[f"fpr_{class_key}"] = row["fpr"]

    return metrics, per_class_df


def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels, sample_indices in dataloader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        sample_indices = sample_indices.to(DEVICE, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels, sample_indices)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / max(total, 1), correct / max(total, 1)


def val_step(model, dataloader, loss_fn, num_classes: int):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    y_true_all = []
    y_prob_all = []
    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(DEVICE, non_blocking=PIN_MEMORY)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            y_true_all.append(labels.cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_prob = np.concatenate(y_prob_all) if y_prob_all else np.array([])
    val_auc = compute_auc(y_true, y_prob, num_classes) if y_true.size else float("nan")
    return running_loss / max(total, 1), correct / max(total, 1), val_auc


def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, patience, num_classes: int):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": []}
    best_val_auc = -np.inf
    best_state = copy.deepcopy(model.state_dict())
    patience_count = 0
    for epoch in tqdm(range(epochs), leave=False):
        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc, val_auc = val_step(model, val_loader, loss_fn, num_classes)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_auc={val_auc:.4f}"
        )
        improved = val_auc > best_val_auc if not np.isnan(val_auc) else False
        if improved:
            print(" Validation AUC improved, saving best model...")
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break
    model.load_state_dict(best_state)
    return history


def append_row(csv_path, row):
    old_df = read_csv_if_exists(csv_path)
    new_df = pd.DataFrame([row])
    if old_df.empty:
        out_df = new_df
    else:
        cols = list(dict.fromkeys(list(old_df.columns) + list(new_df.columns)))
        out_df = pd.concat([old_df.reindex(columns=cols), new_df.reindex(columns=cols)], ignore_index=True)
    out_df.to_csv(csv_path, index=False, lineterminator="\n")


def read_csv_if_exists(csv_path):
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()
def compute_averages_from_experiments(exp_df):
    if exp_df.empty:
        return pd.DataFrame()
    ok_df = exp_df[exp_df["status"] == "ok"].copy()
    if ok_df.empty:
        return pd.DataFrame()
    preferred_group_cols = ["model", "task", "sensor", "hp_tag", "lr", "batch_size", "beta"]
    group_cols = [col for col in preferred_group_cols if col in ok_df.columns]
    excluded = {"run_id", "seed", "status", "model_path", "report_path", "best_epoch", "best_val_auc", "error"}
    metric_cols = [col for col in ok_df.columns if col not in group_cols and col not in excluded]
    return ok_df.groupby(group_cols, dropna=False).agg(n_runs_completed=("run_id", "count"), **{col: (col, "mean") for col in metric_cols}).reset_index()


def df_to_md(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


def write_run_report(path, run_id, model_name, task, sensor, lr, batch_size, beta, seed, best_epoch, best_val_auc, model_path, val_metrics, test_metrics, val_class_df, test_class_df):
    global_df = pd.DataFrame([
        {"Split": "Validation", "Accuracy": val_metrics["accuracy"], "F1": val_metrics["f1"], "AUC": val_metrics["auc"], "G": val_metrics["G"], "Fairness-Weighted Objective": val_metrics["metric_A"], "Per-Class Accuracy Improvement Rate": val_metrics["metric_B"], "Improved Classes": val_metrics["metric_B_n_improved"], "Mean Class Accuracy Delta": val_metrics["metric_B_mean_delta"], "Equalized Odds Gap (skin)": val_metrics["eo_gap_skin"], "Equalized Odds Gap (sex)": val_metrics["eo_gap_sex"], "Equalized Odds Gap (age)": val_metrics["eo_gap_age"]},
        {"Split": "Test", "Accuracy": test_metrics["accuracy"], "F1": test_metrics["f1"], "AUC": test_metrics["auc"], "G": test_metrics["G"], "Fairness-Weighted Objective": test_metrics["metric_A"], "Per-Class Accuracy Improvement Rate": test_metrics["metric_B"], "Improved Classes": test_metrics["metric_B_n_improved"], "Mean Class Accuracy Delta": test_metrics["metric_B_mean_delta"], "Equalized Odds Gap (skin)": test_metrics["eo_gap_skin"], "Equalized Odds Gap (sex)": test_metrics["eo_gap_sex"], "Equalized Odds Gap (age)": test_metrics["eo_gap_age"]},
    ])
    text = f"""# Run {run_id}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **beta**: {beta}
- **alpha**: {ALPHA}
- **seed**: {seed}
- **best_epoch**: {best_epoch}
- **best_val_auc**: {best_val_auc:.6f}
- **model_path**: {model_path}

## Global metrics
{df_to_md(global_df)}

## Validation per-class metrics
{df_to_md(val_class_df)}

## Test per-class metrics
{df_to_md(test_class_df)}
"""
    path.write_text(text, encoding="utf-8")


def write_summary_report(path, model_name, task, sensor, hp_tag, lr, batch_size, beta, avg_row):
    metric_rows = []
    for metric in ["accuracy", "f1", "auc", "G", "metric_A", "metric_B", "metric_B_n_improved", "metric_B_mean_delta", "eo_gap_skin", "eo_gap_sex", "eo_gap_age"]:
        metric_rows.append({"Metric": metric, "Validation Mean": float(avg_row.get(f"val_{metric}", np.nan)), "Test Mean": float(avg_row.get(f"test_{metric}", np.nan))})
    val_class_rows = [{"class": col.replace("val_acc_", "", 1), "accuracy_mean": float(avg_row[col])} for col in avg_row.index if col.startswith("val_acc_")]
    test_class_rows = [{"class": col.replace("test_acc_", "", 1), "accuracy_mean": float(avg_row[col])} for col in avg_row.index if col.startswith("test_acc_")]
    text = f"""# Summary {hp_tag}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **beta**: {beta}
- **alpha**: {ALPHA}
- **n_runs_completed**: {int(avg_row['n_runs_completed'])}

## Global metrics (mean)
{df_to_md(pd.DataFrame(metric_rows))}

## Validation per-class metrics (mean)
{df_to_md(pd.DataFrame(val_class_rows))}

## Test per-class metrics (mean)
{df_to_md(pd.DataFrame(test_class_rows))}
"""
    path.write_text(text, encoding="utf-8")


def build_weight_summary_table(task, sensor):
    train_data, _, _, _ = load_datasets(task, sensor)
    tables = []
    for beta in BETA_VALUES:
        _, _, _, combo_df = compute_class_sensitive_weights(train_data, beta)
        combo_df = combo_df.copy()
        combo_df.insert(0, "Task", task)
        combo_df.insert(1, "Sensor", sensor)
        combo_df.insert(2, "Beta", beta)
        tables.append(
            combo_df[["Task", "Sensor", "Beta", "class", "skin_tone", "sex", "age", "class_count", "combo_count", "combo_weight"]].rename(columns={
                "class": "Class",
                "skin_tone": "Skin Tone",
                "sex": "Sex",
                "age": "Age",
                "class_count": "Class Count",
                "combo_count": "Combo Count",
                "combo_weight": "Weight",
            })
        )
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


BASELINE_11F_CLASS_ACCURACY = load_baseline_11f_per_class_accuracy()
if BASELINE_11F_CLASS_ACCURACY is None:
    print("Warning: Experiment 1 baseline for 11f dermoscopic ViT was not found. The per-class metrics improvement rate will remain NaN.")
else:
    print("Experiment 1 baseline for 11f dermoscopic ViT loaded successfully.")

for task in TASKS_TO_RUN:
    weight_table = build_weight_summary_table(task, "dermoscopic")
    print(f"\nClass x skin_tone x sex x age weights for {task} | dermoscopic")
    if display is None:
        print(weight_table.to_string(index=False))
    else:
        display(weight_table)


## 8. Full Training Loop

This section runs the restricted ViT dermoscopic experiment grid over the selected `beta` values for `CB-CrossEntropy`.


In [ ]:
for model_name in MODELS_TO_RUN:
    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            if (task, sensor) not in DATASET_ROOTS:
                continue
            train_data, val_data, test_data, class_names = load_datasets(task, sensor)
            n_runs = N_RUNS_BY_MODEL[model_name]
            combo_models_root = MODELS_ROOT / model_name / task / sensor
            combo_reports_root = REPORTS_ROOT / model_name / task / sensor
            combo_runs_root = RUNS_ROOT / model_name / task / sensor
            for path in [combo_models_root, combo_reports_root, combo_runs_root]:
                path.mkdir(parents=True, exist_ok=True)
            experiments_csv = combo_runs_root / "experiments.csv"
            averages_csv = combo_runs_root / "averages.csv"
            print("\n" + "=" * 80)
            print(f"{model_name} | {task} | {sensor} | runs={n_runs}")
            print("=" * 80)
            baseline_val = BASELINE_11F_CLASS_ACCURACY["val"] if task == "11f" and BASELINE_11F_CLASS_ACCURACY is not None else None
            baseline_test = BASELINE_11F_CLASS_ACCURACY["test"] if task == "11f" and BASELINE_11F_CLASS_ACCURACY is not None else None
            dataloaders_by_batch_size = {
                batch_size: make_dataloaders(train_data, val_data, test_data, batch_size=batch_size)
                for batch_size in BATCH_SIZES_BY_TASK[task]
            }
            exp_df = read_csv_if_exists(experiments_csv)
            completed_run_ids = set()
            if RESUME and not exp_df.empty and "run_id" in exp_df.columns and "status" in exp_df.columns:
                completed_run_ids = set(exp_df.loc[exp_df["status"] == "ok", "run_id"].astype(str).tolist())
            for lr in LRS_BY_MODEL[model_name]:
                for batch_size in BATCH_SIZES_BY_TASK[task]:
                    train_loader, val_loader, test_loader = dataloaders_by_batch_size[batch_size]
                    for beta in BETA_VALUES_BY_TASK[task]:
                        hp_tag = f"lr{format_lr(lr)}_bs{batch_size}_beta{format_float_tag(beta)}"
                        hp_model_dir = combo_models_root / hp_tag
                        hp_report_dir = combo_reports_root / hp_tag
                        hp_model_dir.mkdir(parents=True, exist_ok=True)
                        hp_report_dir.mkdir(parents=True, exist_ok=True)
                        print(f"\n--- {hp_tag} ---")
                        weight_cache_key = (task, sensor, float(beta))
                        loss_fn, class_counts, class_sensitive_weight_df, sample_weights, preview_weight_df = create_cb_sensitive_cross_entropy_loss(train_data, beta=beta, cache_key=weight_cache_key)
                        for run_idx in range(1, n_runs + 1):
                            run_seed = BASE_SEED + run_idx
                            run_id = f"{model_name}_{task}_{sensor}_{hp_tag}_run{run_idx:02d}"
                            model_path = hp_model_dir / f"{run_id}.pth"
                            report_path = hp_report_dir / f"{run_id}.md"
                            if RESUME and run_id in completed_run_ids and model_path.exists():
                                print(f"Skipping: {run_id}")
                                continue
                            set_all_seeds(run_seed)
                            model = None
                            try:
                                model = create_model(model_name, num_classes=len(class_names))
                                optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
                                history = train_model(model, train_loader, val_loader, optimizer, loss_fn, EPOCHS, EARLY_STOPPING_PATIENCE_BY_MODEL[model_name], num_classes=len(class_names))
                                val_metrics, val_class_df = evaluate_split(model, val_loader, val_data, class_names, task, baseline_val)
                                test_metrics, test_class_df = evaluate_split(model, test_loader, test_data, class_names, task, baseline_test)
                                torch.save({"state_dict": model.state_dict(), "model_name": model_name, "task": task, "sensor": sensor, "num_classes": len(class_names), "class_names": class_names, "lr": lr, "batch_size": batch_size, "beta": beta, "class_counts": class_counts.tolist(), "class_sensitive_weight_table": class_sensitive_weight_df.to_dict(orient="records"), "run_id": run_id}, model_path)
                                best_epoch = int(np.nanargmax(history["val_auc"])) + 1 if history["val_auc"] else 0
                                best_val_auc = float(np.nanmax(history["val_auc"])) if history["val_auc"] else np.nan
                                row = {"model": model_name, "task": task, "sensor": sensor, "hp_tag": hp_tag, "run_id": run_id, "seed": run_seed, "lr": lr, "batch_size": batch_size, "beta": beta, "status": "ok", "model_path": str(model_path), "report_path": str(report_path), "best_epoch": best_epoch, "best_val_auc": best_val_auc, "val_accuracy": val_metrics["accuracy"], "val_f1": val_metrics["f1"], "val_auc": val_metrics["auc"], "val_avg_tpr": val_metrics["avg_tpr"], "val_avg_fpr": val_metrics["avg_fpr"], "val_G": val_metrics["G"], "val_objective": val_metrics["objective"], "val_metric_A": val_metrics["metric_A"], "val_metric_B": val_metrics["metric_B"], "val_metric_B_n_improved": val_metrics["metric_B_n_improved"], "val_metric_B_mean_delta": val_metrics["metric_B_mean_delta"], "val_eo_gap_skin": val_metrics["eo_gap_skin"], "val_eo_gap_sex": val_metrics["eo_gap_sex"], "val_eo_gap_age": val_metrics["eo_gap_age"], "test_accuracy": test_metrics["accuracy"], "test_f1": test_metrics["f1"], "test_auc": test_metrics["auc"], "test_avg_tpr": test_metrics["avg_tpr"], "test_avg_fpr": test_metrics["avg_fpr"], "test_G": test_metrics["G"], "test_objective": test_metrics["objective"], "test_metric_A": test_metrics["metric_A"], "test_metric_B": test_metrics["metric_B"], "test_metric_B_n_improved": test_metrics["metric_B_n_improved"], "test_metric_B_mean_delta": test_metrics["metric_B_mean_delta"], "test_eo_gap_skin": test_metrics["eo_gap_skin"], "test_eo_gap_sex": test_metrics["eo_gap_sex"], "test_eo_gap_age": test_metrics["eo_gap_age"]}
                                for key, value in val_metrics.items():
                                    if key.startswith("acc_") or key.startswith("tpr_") or key.startswith("fpr_"):
                                        row[f"val_{key}"] = value
                                for key, value in test_metrics.items():
                                    if key.startswith("acc_") or key.startswith("tpr_") or key.startswith("fpr_"):
                                        row[f"test_{key}"] = value
                                append_row(experiments_csv, row)
                                completed_run_ids.add(run_id)
                                write_run_report(report_path, run_id, model_name, task, sensor, lr, batch_size, beta, run_seed, best_epoch, best_val_auc, model_path, val_metrics, test_metrics, val_class_df, test_class_df)
                                print(f"run {run_idx:02d}/{n_runs} | val_acc={val_metrics['accuracy']:.4f} test_acc={test_metrics['accuracy']:.4f} test_auc={test_metrics['auc']:.4f} test_metric_A={test_metrics['metric_A']:.4f} test_metric_B={test_metrics['metric_B']:.4f}")
                            except Exception as error:
                                append_row(experiments_csv, {"model": model_name, "task": task, "sensor": sensor, "hp_tag": hp_tag, "run_id": run_id, "seed": run_seed, "lr": lr, "batch_size": batch_size, "beta": beta, "status": "fail", "error": f"{type(error).__name__}: {error}"})
                                print(f"ERROR in {run_id}: {error}")
                            finally:
                                cleanup_memory(model)
                        hp_exp_df = read_csv_if_exists(experiments_csv)
                        hp_avg_df = compute_averages_from_experiments(hp_exp_df)
                        if not hp_avg_df.empty:
                            hp_avg_df.to_csv(averages_csv, index=False)
                            hp_row = hp_avg_df[hp_avg_df["hp_tag"] == hp_tag]
                            if not hp_row.empty and int(hp_row.iloc[0]["n_runs_completed"]) >= n_runs:
                                write_summary_report(hp_report_dir / "summary.md", model_name, task, sensor, hp_tag, lr, batch_size, beta, hp_row.iloc[0])
                                print(f"summary.md updated for {hp_tag}")
            cleanup_memory()
print("Training pipeline finished.")


## 9. Final Consolidation

This section rebuilds `averages.csv`, `top5_summary.csv`, and `top5_summary.md` from the run-level results.


In [ ]:
def build_top5_table(avg_df, top_k=5):
    if avg_df.empty:
        return pd.DataFrame()
    sort_cols = [col for col in ["test_metric_A", "test_auc", "val_metric_A"] if col in avg_df.columns]
    if not sort_cols:
        return pd.DataFrame()
    top_df = avg_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(top_k).copy()
    core_cols = [
        "model", "task", "sensor", "hp_tag", "lr", "batch_size", "beta", "n_runs_completed",
        "val_accuracy", "val_f1", "val_auc", "val_G", "val_metric_A", "val_metric_B", "val_metric_B_n_improved", "val_metric_B_mean_delta", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age",
        "test_accuracy", "test_f1", "test_auc", "test_G", "test_metric_A", "test_metric_B", "test_metric_B_n_improved", "test_metric_B_mean_delta", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age",
    ]
    class_cols = sorted([col for col in top_df.columns if col.startswith("val_acc_") or col.startswith("test_acc_") or col.startswith("val_tpr_") or col.startswith("test_tpr_") or col.startswith("val_fpr_") or col.startswith("test_fpr_") or col.startswith("val_tpr_") or col.startswith("test_tpr_") or col.startswith("val_fpr_") or col.startswith("test_fpr_")])
    cols = [col for col in core_cols + class_cols if col in top_df.columns]
    return top_df[cols]


for model_name in MODELS_TO_RUN:
    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            run_dir = RUNS_ROOT / model_name / task / sensor
            if not run_dir.exists():
                continue
            experiments_csv = run_dir / "experiments.csv"
            averages_csv = run_dir / "averages.csv"
            top5_csv = run_dir / "top5_summary.csv"
            top5_md = run_dir / "top5_summary.md"
            exp_df = read_csv_if_exists(experiments_csv)
            if exp_df.empty:
                continue
            avg_df = compute_averages_from_experiments(exp_df)
            if avg_df.empty:
                continue
            avg_df.to_csv(averages_csv, index=False)
            top5_df = build_top5_table(avg_df, top_k=5)
            top5_df.to_csv(top5_csv, index=False)
            md_text = f"""# Top-5 Summary ({model_name} | {task} | {sensor})

- Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- Source: {averages_csv}
- Ranking: test_metric_A, test_auc, val_metric_A

{df_to_md(top5_df)}
"""
            top5_md.write_text(md_text, encoding="utf-8")
            print(f"Updated: {run_dir}")

print("Final consolidation complete.")


## 10. Minimal Execution Order

The minimal execution order for this notebook is:

1. run the imports and global configuration cells;
2. run metadata preparation, dataset loading, and model factory;
3. run the custom-loss training and evaluation cells;
4. run the full training loop;
5. run the consolidation, heatmaps, and final summary cells.


## 11. Per-Class Accuracy Comparison and Final Selection

The next cells summarise the custom-loss experiment without retraining the models.

Two selection criteria are reported:
- `FP Score`: the fairness-weighted objective already used in the rest of the project.
- `Per-Class Accuracy Improvement Rate`: proportion of classes whose per-class accuracy is higher than the corresponding Experiment 1 baseline.

Additional per-class summary columns are also reported:
- `Improved Classes`: number of classes whose per-class accuracy improves with respect to the baseline.
- `Improved Class List`: classes whose per-class accuracy improves with respect to the baseline.
- `Mean Class Accuracy Delta`: average difference between the current per-class accuracies and the baseline per-class accuracies.

The classwise comparison tables below are shown separately for each task so they are easier to read.


In [ ]:
BASELINE_CONFIGS = {
    "11f": {"model": "ViT", "sensor": "dermoscopic", "lr": 3e-5, "batch_size": 32},
    "2f": {"model": "ViT", "sensor": "dermoscopic", "lr": 3e-5, "batch_size": 16},
}


def load_average_row(root_dir, task, sensor, model_name, lr, batch_size):
    avg_path = root_dir / "runs" / model_name / task / sensor / "averages.csv"
    if not avg_path.exists():
        return None
    avg_df = pd.read_csv(avg_path)
    if avg_df.empty:
        return None
    avg_df["lr"] = pd.to_numeric(avg_df["lr"], errors="coerce")
    avg_df["batch_size"] = pd.to_numeric(avg_df["batch_size"], errors="coerce")
    row = avg_df[(np.isclose(avg_df["lr"], lr)) & (avg_df["batch_size"] == batch_size)]
    if row.empty:
        return None
    return row.iloc[0]


def extract_class_suffixes(df, split):
    prefix = f"{split}_acc_"
    return [col[len(prefix):] for col in df.columns if col.startswith(prefix)]


def suffix_to_label(suffix):
    if suffix in {"benign", "malignant"}:
        return suffix.capitalize()
    return suffix.upper()


def get_objective_value(row, split):
    for col in [f"{split}_objective", f"{split}_metric_A"]:
        if col in row.index and not pd.isna(row[col]):
            return float(row[col])
    return float("nan")


def build_classwise_metrics(avg_row, baseline_row, class_suffixes, split):
    improved_labels = []
    deltas = []
    for suffix in class_suffixes:
        current_value = float(avg_row.get(f"{split}_acc_{suffix}", np.nan))
        baseline_value = float(baseline_row.get(f"{split}_acc_{suffix}", np.nan))
        if np.isnan(current_value) or np.isnan(baseline_value):
            continue
        delta = current_value - baseline_value
        deltas.append(delta)
        if delta > 0:
            improved_labels.append(suffix_to_label(suffix))
    if not deltas:
        return {
            "improvement_rate": np.nan,
            "improved_count": np.nan,
            "improved_list": "",
            "mean_delta": np.nan,
        }
    return {
        "improvement_rate": float(len(improved_labels) / len(deltas)),
        "improved_count": int(len(improved_labels)),
        "improved_list": ", ".join(improved_labels),
        "mean_delta": float(np.mean(deltas)),
    }


def build_scored_custom_df(task):
    config = BASELINE_CONFIGS[task]
    baseline_row = load_average_row(
        Path("Experiment_1") / "training_runs_final",
        task,
        config["sensor"],
        config["model"],
        config["lr"],
        config["batch_size"],
    )
    if baseline_row is None:
        return pd.DataFrame(), None, []

    avg_path = RUNS_ROOT / "ViT" / task / "dermoscopic" / "averages.csv"
    if not avg_path.exists():
        return pd.DataFrame(), baseline_row, []

    custom_df = pd.read_csv(avg_path)
    if custom_df.empty:
        return pd.DataFrame(), baseline_row, []

    custom_df["beta"] = pd.to_numeric(custom_df["beta"], errors="coerce")
    custom_df["lr"] = pd.to_numeric(custom_df["lr"], errors="coerce")
    custom_df["batch_size"] = pd.to_numeric(custom_df["batch_size"], errors="coerce")

    class_suffixes = extract_class_suffixes(custom_df, "val")

    val_rates, val_counts, val_lists, val_means = [], [], [], []
    test_rates, test_counts, test_lists, test_means = [], [], [], []
    val_objectives, test_objectives = [], []

    for _, row in custom_df.iterrows():
        val_metrics = build_classwise_metrics(row, baseline_row, class_suffixes, "val")
        test_metrics = build_classwise_metrics(row, baseline_row, class_suffixes, "test")
        val_rates.append(val_metrics["improvement_rate"])
        val_counts.append(val_metrics["improved_count"])
        val_lists.append(val_metrics["improved_list"])
        val_means.append(val_metrics["mean_delta"])
        test_rates.append(test_metrics["improvement_rate"])
        test_counts.append(test_metrics["improved_count"])
        test_lists.append(test_metrics["improved_list"])
        test_means.append(test_metrics["mean_delta"])
        val_objectives.append(get_objective_value(row, "val"))
        test_objectives.append(get_objective_value(row, "test"))

    custom_df["val_per_class_improvement_rate"] = val_rates
    custom_df["val_improved_classes"] = val_counts
    custom_df["val_improved_class_list"] = val_lists
    custom_df["val_mean_class_accuracy_delta"] = val_means
    custom_df["val_objective_named"] = val_objectives

    custom_df["test_per_class_improvement_rate"] = test_rates
    custom_df["test_improved_classes"] = test_counts
    custom_df["test_improved_class_list"] = test_lists
    custom_df["test_mean_class_accuracy_delta"] = test_means
    custom_df["test_objective_named"] = test_objectives

    return custom_df, baseline_row, class_suffixes


def select_best_beta_by_objective(task):
    custom_df, baseline_row, class_suffixes = build_scored_custom_df(task)
    if custom_df.empty:
        return None, baseline_row, class_suffixes
    sort_cols = ["val_objective_named", "val_auc", "val_accuracy"]
    best_row = custom_df.sort_values(sort_cols, ascending=[False, False, False]).iloc[0]
    return best_row, baseline_row, class_suffixes


def select_best_beta_by_per_class_improvement(task):
    custom_df, baseline_row, class_suffixes = build_scored_custom_df(task)
    if custom_df.empty:
        return None, baseline_row, class_suffixes
    sort_cols = ["val_per_class_improvement_rate", "val_mean_class_accuracy_delta", "val_objective_named", "val_auc"]
    best_row = custom_df.sort_values(sort_cols, ascending=[False, False, False, False]).iloc[0]
    return best_row, baseline_row, class_suffixes


def build_class_accuracy_comparison_table(split, task):
    custom_df, baseline_row, class_suffixes = build_scored_custom_df(task)
    if custom_df.empty or baseline_row is None:
        return pd.DataFrame()
    rows = []
    for _, row in custom_df.sort_values("beta").iterrows():
        for suffix in class_suffixes:
            current_value = float(row.get(f"{split}_acc_{suffix}", np.nan))
            baseline_value = float(baseline_row.get(f"{split}_acc_{suffix}", np.nan))
            if np.isnan(current_value) or np.isnan(baseline_value):
                continue
            delta = current_value - baseline_value
            rows.append({
                "Task": task,
                "Beta": float(row["beta"]),
                "Class": suffix_to_label(suffix),
                "Baseline Accuracy": baseline_value,
                "Custom-Loss Accuracy": current_value,
                "Delta": delta,
                "Improved": delta > 0,
            })
    return pd.DataFrame(rows)


def display_class_accuracy_comparison_tables(split):
    split_label = "Validation" if split == "val" else "Test"
    for task in TASKS_TO_RUN:
        table = build_class_accuracy_comparison_table(split, task)
        if table.empty:
            continue
        print(f"{split_label} per-class metrics comparison | {task} | dermoscopic | ViT")
        display(table)


def build_best_beta_table(selector_fn, include_per_class_columns=False):
    rows = []
    for task in TASKS_TO_RUN:
        best_row, baseline_row, class_suffixes = selector_fn(task)
        if best_row is None:
            continue
        for split, metric_label in [("val", "Validation"), ("test", "Test")]:
            row = {
                "Task": task,
                "Sensor": "dermoscopic",
                "Model": "ViT",
                "LR": float(best_row["lr"]),
                "Batch Size": int(best_row["batch_size"]),
                "Beta": float(best_row["beta"]),
                "Accuracy": float(best_row[f"{split}_accuracy"]),
                "AUC": float(best_row[f"{split}_auc"]),
                "Equalized Odds Gap (Skin)": float(best_row[f"{split}_eo_gap_skin"]),
                "Equalized Odds Gap (Sex)": float(best_row[f"{split}_eo_gap_sex"]),
                "Equalized Odds Gap (Age)": float(best_row[f"{split}_eo_gap_age"]),
                "G": float(best_row[f"{split}_G"]),
                "Objective": float(get_objective_value(best_row, split)),
                "Metric": metric_label,
            }
            if include_per_class_columns:
                row.update({
                    "Per-Class Accuracy Improvement Rate": float(best_row[f"{split}_per_class_improvement_rate"]),
                    "Improved Classes": int(best_row[f"{split}_improved_classes"]),
                    "Improved Class List": best_row[f"{split}_improved_class_list"],
                    "Mean Class Accuracy Delta": float(best_row[f"{split}_mean_class_accuracy_delta"]),
                })
            rows.append(row)
    return pd.DataFrame(rows)


## 12. Validation and Test Per-Class Accuracy Tables

The next two cells display the per-class accuracy comparisons separately for `11f` and `2f` in validation and test.


In [ ]:
display_class_accuracy_comparison_tables("val")


In [ ]:
display_class_accuracy_comparison_tables("test")


## 13. Best Beta Selected By FP Score

This table selects the best `beta` for each task using the **validation** `FP Score`, and then reports both validation and test results.


In [ ]:
best_beta_by_objective_table = build_best_beta_table(
    selector_fn=select_best_beta_by_objective,
    include_per_class_columns=False,
)
print("Best beta selected by validation Objective")
display(best_beta_by_objective_table)


## 14. Best Beta Selected By Per-Class Accuracy Improvement

This final table has the same structure as the previous summary, but it also includes the per-class accuracy improvement information and selects the best `beta` using the **validation** `Per-Class Accuracy Improvement Rate`.


In [ ]:
best_beta_by_per_class_accuracy_table = build_best_beta_table(
    selector_fn=select_best_beta_by_per_class_improvement,
    include_per_class_columns=True,
)
print("Best beta selected by validation per-class metrics improvement")
display(best_beta_by_per_class_accuracy_table)


## 15. Final Class-Sensitive Weights For The Winning Combinations

This final table reports the final `class x skin_tone` weights induced by the winning `beta` for each task when the winner is selected by **validation FP Score**.


In [ ]:
def build_final_weight_tables_for_per_class_winners():
    tables = {}
    for task in TASKS_TO_RUN:
        best_row, _, _ = select_best_beta_by_per_class_improvement(task)
        if best_row is None:
            continue
        beta = float(best_row["beta"])
        train_data, _, _, _ = load_datasets(task, "dermoscopic")
        _, _, _, combo_df = compute_class_sensitive_weights(train_data, beta=beta)
        if combo_df.empty:
            continue
        combo_df = combo_df.copy()
        combo_df.insert(0, "Task", task)
        combo_df.insert(1, "Beta", beta)
        tables[task] = combo_df[[
            "Task", "Beta", "class", "skin_tone", "sex", "age", "class_count", "combo_count", "combo_weight"
        ]].rename(columns={
            "class": "Class",
            "skin_tone": "Skin Tone",
            "sex": "Sex",
            "age": "Age",
            "class_count": "Class Count",
            "combo_count": "Combo Count",
            "combo_weight": "Final Weight",
        })
    return tables


final_weight_tables = build_final_weight_tables_for_per_class_winners()
for task in TASKS_TO_RUN:
    if task not in final_weight_tables:
        continue
    print(f"Final class-sensitive weights | winner selected by per-class metrics improvement | {task} | dermoscopic | ViT")
    display(final_weight_tables[task])


## Updated TPR/FPR Summary Tables

These tables read the saved `averages.csv` files and display the best configurations selected by validation `FP Score`. They include average TPR/FPR and a separate per-class TPR/FPR table.


In [ ]:
def _read_csv_if_exists(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()


def _resolve_runs_root_for_tpr_fpr_tables():
    if "RUNS_ROOT" in globals():
        return Path(RUNS_ROOT)

    project_root = Path.cwd().resolve()
    while project_root != project_root.parent and not (project_root / "data" / "raw").exists():
        project_root = project_root.parent
    if not (project_root / "data" / "raw").exists():
        project_root = Path.cwd().resolve()

    candidates = [
        project_root / "Experiment_1" / "training_runs_final_fixed" / "runs",
                project_root / "Experiment_2" / "TR_fixed_skin" / "runs",
        project_root / "Experiment_2" / "TR_fixed" / "runs",
        project_root / "Experiment_3" / "TR_fixed" / "runs",
    ]
    existing = [path for path in candidates if path.exists()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        print("RUNS_ROOT was not defined. Using first existing candidate:", existing[0])
        return existing[0]
    raise NameError("RUNS_ROOT is not defined and no default runs directory was found.")


def _load_all_average_results():
    rows = []
    runs_root = _resolve_runs_root_for_tpr_fpr_tables()
    for avg_path in sorted(runs_root.rglob("averages.csv")):
        df = _read_csv_if_exists(avg_path)
        if df.empty:
            continue
        df = df.copy()
        df["averages_path"] = str(avg_path)
        rows.append(df)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def _numeric_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _select_best_rows_by_validation_objective(avg_df):
    if avg_df.empty or "val_objective" not in avg_df.columns:
        return pd.DataFrame()

    avg_df = avg_df.copy()
    avg_df = _numeric_columns(
        avg_df,
        [
            "lr", "batch_size", "beta", "val_objective", "val_auc", "test_objective", "test_auc",
            "val_accuracy", "test_accuracy", "val_avg_tpr", "test_avg_tpr", "val_avg_fpr", "test_avg_fpr",
            "val_G", "test_G", "val_eo_gap_skin", "test_eo_gap_skin", "val_eo_gap_sex", "test_eo_gap_sex",
            "val_eo_gap_age", "test_eo_gap_age",
        ],
    )

    group_cols = [col for col in ["task", "sensor"] if col in avg_df.columns]
    if not group_cols:
        return avg_df.sort_values(["val_objective", "val_auc"], ascending=[False, False]).head(1)

    selected = []
    sort_cols = [col for col in ["val_objective", "val_auc", "test_objective", "test_auc"] if col in avg_df.columns]
    for _, group_df in avg_df.groupby(group_cols, sort=False):
        selected.append(group_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).iloc[0])
    return pd.DataFrame(selected).reset_index(drop=True)


def _build_best_summary_with_avg_tpr_fpr():
    avg_df = _load_all_average_results()
    best_df = _select_best_rows_by_validation_objective(avg_df)
    if best_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in best_df.iterrows():
        for split, label in [("val", "Validation"), ("test", "Test")]:
            rows.append({
                "Task": row.get("task", ""),
                "Sensor": row.get("sensor", ""),
                "Model": row.get("model", ""),
                "HP Tag": row.get("hp_tag", ""),
                "LR": row.get("lr", np.nan),
                "Batch Size": row.get("batch_size", np.nan),
                "Beta": row.get("beta", np.nan),
                "Metric": label,
                "Accuracy": row.get(f"{split}_accuracy", np.nan),
                "AUC": row.get(f"{split}_auc", np.nan),
                "Average TPR": row.get(f"{split}_avg_tpr", np.nan),
                "Average FPR": row.get(f"{split}_avg_fpr", np.nan),
                "EO Gap (Skin)": row.get(f"{split}_eo_gap_skin", np.nan),
                "EO Gap (Sex)": row.get(f"{split}_eo_gap_sex", np.nan),
                "EO Gap (Age)": row.get(f"{split}_eo_gap_age", np.nan),
                "G": row.get(f"{split}_G", np.nan),
                "Objective": row.get(f"{split}_objective", np.nan),
            })

    out_df = pd.DataFrame(rows)
    if out_df["Beta"].isna().all():
        out_df = out_df.drop(columns=["Beta"])
    return out_df


def _class_label_from_suffix(suffix):
    if suffix in {"benign", "malignant"}:
        return suffix.capitalize()
    return suffix.upper()


def _build_best_per_class_tpr_fpr_table():
    avg_df = _load_all_average_results()
    best_df = _select_best_rows_by_validation_objective(avg_df)
    if best_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in best_df.iterrows():
        for split, label in [("val", "Validation"), ("test", "Test")]:
            prefix = f"{split}_tpr_"
            class_suffixes = sorted([col[len(prefix):] for col in row.index if col.startswith(prefix)])
            for suffix in class_suffixes:
                rows.append({
                    "Task": row.get("task", ""),
                    "Sensor": row.get("sensor", ""),
                    "Model": row.get("model", ""),
                    "HP Tag": row.get("hp_tag", ""),
                    "Beta": row.get("beta", np.nan),
                    "Metric": label,
                    "Class": _class_label_from_suffix(suffix),
                    "TPR": row.get(f"{split}_tpr_{suffix}", np.nan),
                    "FPR": row.get(f"{split}_fpr_{suffix}", np.nan),
                })

    out_df = pd.DataFrame(rows)
    if not out_df.empty and out_df["Beta"].isna().all():
        out_df = out_df.drop(columns=["Beta"])
    return out_df


best_summary_with_avg_tpr_fpr = _build_best_summary_with_avg_tpr_fpr()
print("Best configurations selected by validation Objective, including average TPR/FPR")
display(best_summary_with_avg_tpr_fpr)

best_per_class_tpr_fpr_table = _build_best_per_class_tpr_fpr_table()
print("Per-class TPR/FPR for the best configurations selected by validation Objective")
display(best_per_class_tpr_fpr_table)
